# A04 — customer_behavior: dữ liệu và so sánh đã lưu

Các framework dùng hai môi trường WSL riêng. Mã huấn luyện ở `training/customer_behavior_keras.ipynb` và `training/customer_behavior_torch.ipynb`. Notebook này đọc kết quả gốc; Run All không train/TEST. Toàn bộ notebook huấn luyện nguyên byte và output cũ nằm trong provenance.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Customer Behavior: dự đoán customer churn từ hành vi mua hàng và tương tác của khách. A04 dùng Olist Brazilian E-Commerce Public Dataset đã được chuẩn bị thành customer-level churn dataset trong `datasets/customer_behavior/a04`.


Output cho thấy điều gì?

Bảng overview xác nhận A04 giữ đúng business problem của A03: dự đoán `is_churned` cho khách hàng. Dataset A04 là Olist customer-level churn với 58,101 samples, 29 raw input features và target nhị phân; `is_churned=1` nghĩa là khách không có repeat order hợp lệ trong outcome window 180 ngày.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một prepared customer dataset. CPU vẫn chạy được vì dữ liệu đã aggregate ở cấp customer, không phải event stream thô nhiều triệu dòng.


# 3. Load Dataset

Cell này làm gì?

Liệt kê raw Olist files và đọc prepared customer-level churn CSV. Raw files được giữ nguyên; notebook dùng file prepared để tránh tự tạo target khác A03.


Output cho thấy điều gì?

Folder A04 có raw relational Olist tables và một prepared customer-level file. Notebook đọc `olist_customer_churn_a04.csv` vì file này giữ target `is_churned` đã chốt cho continuity với A03.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra shape, columns, target distribution, missing values, duplicates, descriptive statistics, categorical columns và evidence về sequence/time structure.


Output cho thấy điều gì?

Prepared dataset có 58,101 dòng và 31 cột, không có missing value trong file đã chuẩn bị và không có duplicate row. Target cực kỳ mất cân bằng: class 0 có 665 khách (1.14%), class 1 có 57,436 khách (98.86%); vì vậy notebook báo thêm Macro F1 và Balanced Accuracy để tránh bị accuracy đánh lừa.


# 5. Data Preparation

Cell này làm gì?

Loại identifier, group rare categories theo train only, chia train/validation/test bằng stratify, rồi fit imputer/scaler/one-hot encoder chỉ trên training data.


Output cho thấy điều gì?

Split được stratify thành train/validation/test và preprocessor chỉ fit trên train set. Sau encoding và scaling, mỗi khách hàng được biểu diễn bằng vector tabular gồm 169 features; target test vẫn được giữ riêng cho final evaluation.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN/sequence modeling không.


Output cho thấy điều gì?

Representation chính là customer-level aggregate tabular vector, không phải ảnh và cũng chưa phải event sequence. Raw Olist có timestamped orders/payments/reviews, nhưng notebook này không tự reorder columns để tạo sequence giả; MLP/DNN là baseline tự nhiên, còn Conv1D chỉ được minh họa như operation giáo dục.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


Output cho thấy điều gì?

Dense tạo logits từ vector customer features, sigmoid chuyển logits thành xác suất churn. Conv1D/pooling chỉ minh họa trên toy sequence có thứ tự rõ, không áp đặt lên các cột aggregate của Olist.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP tạo xác suất churn từ customer aggregate features. Conv1D thủ công chỉ giải thích operation trên toy sequence, không dùng làm model chính cho prepared Olist table.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho churn classification.


Output cho thấy điều gì?

Keras baseline nhận vector customer behavior đã transform và trả về xác suất `is_churned=1`. Vì label churn là class đa số, notebook chọn threshold theo Macro F1 để buộc model quan tâm cả retained/non-churn minority class.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


Output cho thấy điều gì?

Training log cho thấy model học trên train và validation qua từng epoch. Class weight làm class 0 hiếm hơn có ảnh hưởng lớn hơn trong loss, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


Output cho thấy điều gì?

Curves giúp nhìn model có học tín hiệu churn beyond class prior hay không. Với imbalance lớn, validation AUC hữu ích vì không phụ thuộc vào một threshold cố định.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng Accuracy, Precision, Recall, F1, Macro F1, Balanced Accuracy, ROC-AUC, confusion matrix và classification report.


Output cho thấy điều gì?

Keras baseline MLP dùng cùng representation tabular và threshold chọn trên validation. Baseline đạt validation F1 0.9916, Macro F1 0.5281, ROC-AUC 0.5915; positive-class F1 rất cao chủ yếu vì churn là class áp đảo, nên Macro F1 mới là tín hiệu quan trọng hơn.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra input/logits shape.


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và logits có dạng `(batch, 1)`, đúng với binary churn classification. PyTorch model giữ cùng ý tưởng hidden layers như Keras baseline.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, weighted BCE cho imbalance, và lưu checkpoint tốt nhất theo validation ROC-AUC.


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, weighted BCE, backward và optimizer step. Vì class 0 rất hiếm, weighted loss giúp model không chỉ tối ưu cho class churn đa số.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng threshold rule. Với Olist, retained/non-churn là minority, nên classification report theo từng class cho biết model có bỏ qua class hiếm hay không.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation metrics.


Output cho thấy điều gì?

Keras và PyTorch dùng kiến trúc baseline tương đương về ý tưởng nhưng runtime khác nhau trên CPU. PyTorch baseline đạt validation Macro F1 0.5308, Keras baseline đạt 0.5281; đây chỉ là một run thực nghiệm, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train làm model chính vì input là aggregate tabular.


Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, threshold rule, seed và training budget. CNN được dừng ở mức investigation vì prepared Olist input không còn event-order structure để Conv1D khai thác.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation Macro F1.


Output cho thấy điều gì?

Trong nhóm architecture experiments, model tốt nhất theo validation Macro F1 là `M4 Residual MLP Feature Attention` với Macro F1 0.5293, F1 0.9919, ROC-AUC 0.5907. Chênh lệch giữa các MLP variant nhỏ, nên lợi ích thực tế của kiến trúc phức tạp cần được đọc cùng runtime và imbalance của target.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


Output cho thấy điều gì?

Hidden activations cho biết model tạo latent representation từ behavior aggregates trước output churn. Attention weights, nếu xem từ M4, là tín hiệu model-inspection trên feature đã transform, không phải nguyên nhân kinh doanh chắc chắn.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation: confusion matrix, false positives, false negatives và high-confidence errors của model được chọn.


Output cho thấy điều gì?

Error analysis trên validation có 8,575 dự đoán đúng, 95 false positives và 45 false negatives. Với churn, false negative nghĩa là model bỏ sót khách có khả năng churn; tuy nhiên do class churn chiếm gần toàn bộ dataset, cần thận trọng khi diễn giải độ tin cậy của từng nhóm lỗi.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation Macro F1, evaluate test set đúng một lần, rồi lưu final model.


Output cho thấy điều gì?

Final model được chọn theo validation Macro F1 và chỉ evaluate test một lần. Model `M4 Residual MLP Feature Attention` dùng threshold 0.20; test Accuracy 0.9841, Precision 0.9890, Recall 0.9949, F1 0.9920, Macro F1 0.5295, ROC-AUC 0.6545.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc metric A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không claim score comparison trực tiếp.


Output cho thấy điều gì?

So sánh A03/A04 cho thấy hai notebook giữ cùng target concept `is_churned`, nhưng dữ liệu khác nguồn và khác cách tạo label. Vì vậy score A03 và A04 không phải benchmark trực tiếp; giá trị chính của A04 là dataset thật hơn, workflow DL đầy đủ hơn và kiểm soát rõ hơn các artifact/representation.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Notebook đã lưu final model, checkpoint, metrics, figures và metadata dưới `pipeline04/customer_behavior`. Tổng runtime train/evaluate được ghi nhận khoảng 118.23 giây; hạn chế lớn nhất còn lại là Olist có repeat purchase rất thấp, khiến target `is_churned` lệch mạnh và cần được xử lý kỹ ở phần báo cáo.


In [ ]:
import pandas as pd
from IPython.display import display, Image
_folder = _repo / 'A04/artifacts/pipeline04/customer_behavior/metrics'
for _file in sorted(_folder.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/customer_behavior/figures/keras_basic_mlp_training_curves.png')))
